# 02 · Resiliencia, Teorema CAP y recuperación

**Fase 2.** Los cortes se provocan con comandos del host (el contenedor Jupyter no controla Docker);
este notebook muestra la evidencia observable y deja cada celda re-ejecutable durante la prueba.

| Escenario | Comando (host) | Qué demuestra |
|---|---|---|
| Corte de energía en Sur | `python tareas.py recuperacion-wal` | WAL: REDO / UNDO |
| Caída de un nodo Cassandra | `python tareas.py caida-cassandra` | AP: `ONE`/`QUORUM` vs `ALL` |
| Partición de red del Norte | `python tareas.py particion-red` | AP en tracking, autonomía local en PostgreSQL |
| Caída del primario de Central | `python tareas.py failover` → `python tareas.py reincorporar` | CP + conmutación con cero pérdida |
| Backups y PITR | `python tareas.py backup`, `python tareas.py backup-incremental`, `python tareas.py restore` | RPO / RTO |

In [ ]:
import uuid

import pandas as pd
from cassandra import ConsistencyLevel as CL
from IPython.display import display

from logistica import tracking
from logistica.conexiones import conectar_pg, sesion_cassandra
from logistica.consultas import consultar

pd.set_option("display.width", 160)

## 1. Teorema CAP en esta arquitectura

* **Tracking (Cassandra) → AP.** Ante una partición cada nodo sigue aceptando lecturas y escrituras
  con consistencia `ONE`; las réplicas convergen después (hinted handoff, read repair,
  `nodetool repair`). Modelo BASE.
* **Facturación (PostgreSQL) → CP.** Con `synchronous_commit = remote_apply` y un standby síncrono
  obligatorio, un COMMIT no se confirma sin quórum: se sacrifica disponibilidad para evitar
  desfasajes financieros. `python tareas.py failover` muestra el `COMMIT` esperando en `SyncRep`.

Nota: en PostgreSQL esa espera ocurre *después* del commit local; lo que se bloquea es la
**confirmación al cliente**, no se declara éxito sin quórum.

In [ ]:
display(
    consultar(
        "centro",
        """
    SELECT name, setting FROM pg_settings WHERE name IN
    ('synchronous_commit', 'synchronous_standby_names', 'synchronized_standby_slots',
     'archive_mode', 'archive_timeout', 'wal_level', 'summarize_wal', 'checkpoint_timeout')
    ORDER BY name""",
    )
)

## 2. Consistencia ajustable en Cassandra (RF = 3)

| Nivel | Réplicas requeridas | Con 1 nodo caído |
|---|---|---|
| `ONE` | 1 | disponible |
| `QUORUM` | 2 | disponible |
| `ALL` | 3 | **no disponible** |

Con los tres nodos arriba todos funcionan. Para ver la diferencia ejecute en el host
`docker compose stop cassandra-sur` y vuelva a correr la celda (`docker compose start cassandra-sur` al terminar).

In [ ]:
sesion = sesion_cassandra()
print("Nodos activos:", [str(h.address) for h in sesion.cluster.metadata.all_hosts() if h.is_up])

paquete = uuid.uuid4()
resultados = []
for nombre in ("ONE", "QUORUM", "ALL"):
    nivel = getattr(CL, nombre)
    try:
        tracking.registrar_evento(
            paquete, "en_transito", "centro", -34.6, -58.4, consistencia=nivel
        )
        resultados.append((nombre, "escritura aceptada"))
    except Exception as error:
        resultados.append((nombre, f"RECHAZADA ({type(error).__name__})"))
display(pd.DataFrame(resultados, columns=["consistencia", "resultado"]))
for evento in tracking.historial_paquete(paquete):
    tracking.eliminar_evento(paquete, evento["ts"], consistencia=CL.ONE)

## 3. Gestor de recuperación: WAL, checkpoints, REDO y UNDO

Todo cambio se escribe primero en el **WAL** (*write-ahead logging*) y se fuerza a disco en el
`COMMIT`; las páginas de datos se escriben después, en segundo plano o en un **checkpoint**.
Tras un corte, el servidor parte del `redo_lsn` del último checkpoint y:

* **REDO**: reaplica los registros del WAL → recupera transacciones confirmadas cuyas páginas
  no habían llegado al disco (Durabilidad).
* **UNDO**: PostgreSQL no usa log de undo; una transacción sin `COMMIT` queda marcada como no
  confirmada y sus versiones de fila son invisibles (MVCC), `VACUUM` las limpia luego.
  El efecto lógico es el mismo: nada de lo no confirmado sobrevive.

`python tareas.py recuperacion-wal` lo prueba con `kill -9` sobre el nodo Sur.

In [ ]:
with conectar_pg("sur", autocommit=True) as c:

    def wal():
        c.execute("SELECT pg_stat_force_next_flush()")  # las estadísticas se publican con demora
        return c.execute(
            "SELECT wal_records, pg_current_wal_lsn() AS lsn FROM pg_stat_wal"
        ).fetchone()

    antes = wal()
    c.execute(
        "INSERT INTO envios_sur (region, cliente_id, sucursal_origen_id, sucursal_destino_id, peso_kg, distancia_km) "
        "SELECT 'sur', 1, 11, 12, 1, 1 FROM generate_series(1, 1000)"
    )
    despues = wal()
    bytes_wal = c.execute(
        "SELECT pg_wal_lsn_diff(%s, %s) AS b", (despues["lsn"], antes["lsn"])
    ).fetchone()["b"]
    print(
        "1000 inserciones generaron",
        despues["wal_records"] - antes["wal_records"],
        "registros de WAL,",
        int(bytes_wal) // 1024,
        "KiB",
    )
    print("LSN antes:", antes["lsn"], "→ después:", despues["lsn"])
    c.execute("DELETE FROM envios_sur WHERE cliente_id = 1 AND distancia_km = 1")

    print(
        "Último checkpoint:",
        c.execute(
            "SELECT checkpoint_lsn, redo_lsn, checkpoint_time FROM pg_control_checkpoint()"
        ).fetchone(),
    )
    c.execute("CHECKPOINT")
    print(
        "Tras CHECKPOINT:  ",
        c.execute(
            "SELECT checkpoint_lsn, redo_lsn, checkpoint_time FROM pg_control_checkpoint()"
        ).fetchone(),
    )
    print(
        "Segmento WAL actual:",
        c.execute("SELECT pg_walfile_name(pg_current_wal_lsn()) AS f").fetchone()["f"],
    )

## 4. Backups: RPO 15 min · RTO 2 h

* **RPO ≤ 15 min** → `archive_mode = on` y `archive_timeout = 900`: cada 15 minutos (o al llenarse
  16 MiB) el segmento de WAL se archiva; el peor caso de pérdida es lo escrito desde el último archivado.
* **Estrategia**: completo semanal (`python tareas.py backup`), incremental horario
  (`python tareas.py backup-incremental`, usa el resumen de WAL) y archivado continuo. Cassandra: snapshots diarios.
* **RTO ≤ 2 h** → restaurar = copiar el último completo, combinar incrementales
  (`pg_combinebackup`) y reproducir WAL hasta el instante pedido (`python tareas.py restore`). El script mide el tiempo
  real; para el volumen de este laboratorio son segundos y escala con el tamaño de la base y el WAL a reproducir.

In [ ]:
display(
    consultar(
        "centro",
        "SELECT archived_count, failed_count, last_archived_wal, last_archived_time FROM pg_stat_archiver",
    )
)